In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("/kaggle/input/datasets/krish97singh/dataset-entry-devcom/train.csv")
df_og = pd.read_csv("/kaggle/input/datasets/krish97singh/dataset-entry-devcom/train.csv")
#creat them into int 
df['VIP'] = df['VIP'].map({True: 1, False: 0}).astype('Int16')
df['Transported'] = df['Transported'].map({True: 1, False: 0}).astype('Int16')
df['CryoSleep'] = df['CryoSleep'].map({True: 1, False: 0}).astype('Int16')
                    

# splitting column
df['Group'] = df['PassengerId'].str.split('_').str[0].astype('Int16')
df['DeckNumber'] = df['Cabin'].str.split('/').str[0]
df['Side'] = df['Cabin'].str.split('/').str[2]


df['CabinNum'] = pd.to_numeric(df['Cabin'].str.split('/').str[1], errors='coerce')

df['CabinRegion'], bin_edges = pd.qcut(
    df['CabinNum'],
    q=5,
    labels=False,
    retbins=True,
    duplicates='drop'
)

df['Side'] = df['Side'].map({'P': 'P', 'S': 'S'})
df['Side'] = df['Side'].fillna('Missing')
df['Side'] = df['Side'].astype(str)



# removing objects 
drop={'PassengerId',"Cabin","Name"}
df=df.drop(drop,axis=1)

#df.info()
#df

In [2]:

#cost related stuff
RandomCost=['RoomService','FoodCourt','ShoppingMall','Spa',"VRDeck"]
for col in RandomCost:
    df[col] = df[col].fillna(0)
    
df['TotalSpend'] = (
    df['RoomService']
    + df['FoodCourt']
    + df['ShoppingMall']
    + df['Spa']
    + df['VRDeck'])

df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)


##
group_side = df.groupby('Group')['Side'].apply(
    lambda x: x.dropna().iloc[0] if x.dropna().shape[0] > 0 else np.nan
)

group_size = df.groupby('Group').transform('size')

mask = df['Side'].isna() & (group_size > 1)

df.loc[mask, 'Side'] = df.loc[mask, 'Group'].map(group_side)


df= df.dropna(subset=["Side"])


In [3]:

# is spending is 0 then put in cryo sleep otherwise not in 0 , not always true but better odds than just giving them 
df.loc[df['TotalSpend'] != 0, 'CryoSleep'] = (
    df.loc[df['TotalSpend'] != 0, 'CryoSleep'].fillna(0)
)

df.loc[
    (df['CryoSleep'].isna()) &
    (df['TotalSpend'] > 0),
    'CryoSleep'
] = 0
deck_mode = (
    df.groupby('DeckNumber')['CryoSleep']
      .agg(lambda x: x.mode()[0])
)

mask = df['CryoSleep'].isna()

df.loc[mask, 'CryoSleep'] = (
    df.loc[mask, 'DeckNumber']
      .map(deck_mode)
)



"""

df['CryoSleep'] = df['CryoSleep'].fillna(1)
"""


# age but by homeplanet median 

df['Age'] = df['Age'].fillna(
    df.groupby('HomePlanet')['Age'].transform('median')
)





# keeping vip same as vip status is rather rare and no relation found with money spent to vip or not , 
#most are of f and g deck so most likely to not be vips 67%
df['VIP'] = df['VIP'].fillna(0)


df['GroupSize'] = df.groupby('Group')['Group'].transform('size')


df['DeckRisk'] = df['DeckNumber'].map({
    'A':0,
    'B':2,
    'C':2,
    'D':0,
    'E':-1,
    'F':0,
    'G':1,
    'T':-2
})

#### for  home planet 

group_homeplanet = df.groupby('Group')['HomePlanet'].apply(
    lambda x: x.dropna().iloc[0]
    if len(x.dropna()) > 0 else np.nan
)

mask = df['HomePlanet'].isna()

df.loc[mask, 'HomePlanet'] = (
    df.loc[mask, 'Group']
      .map(group_homeplanet)
)

# part 2 

mask = df['HomePlanet'].isna()
deck_hp = {
    'A': 'Europa',
    'B': 'Europa',
    'C': 'Europa',
    'G': 'Earth',
    'T': 'Europa'
}

df.loc[mask, 'HomePlanet'] = (
    df.loc[mask, 'DeckNumber']
      .map(deck_hp)
)

#part 3 -4 
deck_mode = (
    df.groupby('DeckNumber')['HomePlanet']
      .agg(lambda x: x.mode()[0])
)

mask = df['HomePlanet'].isna()

df.loc[mask, 'HomePlanet'] = (
    df.loc[mask, 'DeckNumber']
      .map(deck_mode)
)

df['HomePlanet'] = df['HomePlanet'].fillna(
    df['HomePlanet'].mode()[0]
)
df['Deck_Side'] = df['DeckNumber'].astype(str) + "_" + df['Side'].astype(str)

'''
df = pd.get_dummies(df, columns=['Deck_Side'])
df = pd.get_dummies(df, columns=['DeckNumber'], dtype=int)
df = pd.get_dummies(df, columns=['HomePlanet'], dtype=int)
df = pd.get_dummies(df, columns=['Destination'], dtype=int)
'''


df['IsChild'] = (df['Age'] < 13).astype(int)
df['Cryo_NoSpend'] = (
    (df['CryoSleep'].fillna(0) == 1) &
    (df['NoSpend'].fillna(0) == 1)
).astype(int)
df=df.drop("VIP",axis=1)
df =df.drop("Group",axis=1)



In [4]:
cat_features = [
    'HomePlanet',
    'Destination',
    'DeckNumber',
    'Side',
    'Deck_Side'
]


for col in cat_features:
    if col in df.columns:
        df[col] = df[col].fillna('Missing')
        df[col] = df[col].astype(str)

X = df.drop('Transported', axis=1)
y = df['Transported']

In [5]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = CatBoostClassifier(
    iterations=2000,
    depth=7,
    learning_rate=0.03,
    l2_leaf_reg=5,
    loss_function='Logloss',
    eval_metric='Accuracy',
    verbose=200,
    random_seed=42
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 0.7621513	test: 0.7653824	best: 0.7653824 (0)	total: 72.1ms	remaining: 2m 24s
200:	learn: 0.8275812	test: 0.8033353	best: 0.8050604 (193)	total: 2.93s	remaining: 26.2s
400:	learn: 0.8491516	test: 0.8159862	best: 0.8165612 (398)	total: 5.84s	remaining: 23.3s
600:	learn: 0.8701467	test: 0.8211616	best: 0.8211616 (591)	total: 8.79s	remaining: 20.5s
800:	learn: 0.8882657	test: 0.8159862	best: 0.8223117 (602)	total: 11.8s	remaining: 17.6s
1000:	learn: 0.9017831	test: 0.8171363	best: 0.8223117 (602)	total: 14.8s	remaining: 14.8s
1200:	learn: 0.9153005	test: 0.8194365	best: 0.8223117 (602)	total: 17.8s	remaining: 11.9s
1400:	learn: 0.9252229	test: 0.8211616	best: 0.8223117 (602)	total: 20.8s	remaining: 8.91s
1600:	learn: 0.9342824	test: 0.8217366	best: 0.8223117 (602)	total: 23.9s	remaining: 5.97s
1800:	learn: 0.9413287	test: 0.8217366	best: 0.8228867 (1655)	total: 27s	remaining: 2.98s
1999:	learn: 0.9488064	test: 0.8182864	best: 0.8228867 (1655)	total: 30s	remaining: 0us

bestTest 

CatBoostClassifier(depth=7, eval_metric='Accuracy', iterations=2000, l2_leaf_reg=5, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=200)

In [6]:
y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

import pandas as pd

fi = pd.DataFrame({
    'feature': X.columns,
    'importance': model.get_feature_importance()
}).sort_values(by='importance', ascending=False)

print(fi)

Accuracy: 0.8228867165037378
              precision    recall  f1-score   support

         0.0       0.82      0.82      0.82       863
         1.0       0.82      0.83      0.82       876

    accuracy                           0.82      1739
   macro avg       0.82      0.82      0.82      1739
weighted avg       0.82      0.82      0.82      1739

         feature  importance
11      CabinNum   10.332287
0     HomePlanet    9.486832
7            Spa    8.803044
13    TotalSpend    7.829236
8         VRDeck    7.672904
3            Age    7.177974
17     Deck_Side    6.634714
5      FoodCourt    6.054689
9     DeckNumber    5.395997
4    RoomService    5.349986
10          Side    4.663067
6   ShoppingMall    4.461151
2    Destination    3.712597
19  Cryo_NoSpend    2.911662
1      CryoSleep    2.576156
15     GroupSize    1.932938
14       NoSpend    1.703034
16      DeckRisk    1.578535
12   CabinRegion    1.400231
18       IsChild    0.322965


In [7]:
import pandas as pd

# =========================
# 1. LOAD TEST
# =========================
test_df = pd.read_csv("/kaggle/input/datasets/krish97singh/dataset-entry-devcom/test.csv")
test_raw = test_df.copy()

# =========================
# 2. APPLY SAME FEATURE ENGINEERING
# =========================

test_df['Group'] = test_df['PassengerId'].str.split('_').str[0]
test_df['DeckNumber'] = test_df['Cabin'].str.split('/').str[0]
test_df['Side'] = test_df['Cabin'].str.split('/').str[2]

#df['Side'] = df['Side'].map({'P': 'P', 'S': 'S'})
df['Side'] = df['Side'].fillna('Missing')

# spending
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
for c in spend_cols:
    test_df[c] = test_df[c].fillna(0)

test_df['TotalSpend'] = test_df[spend_cols].sum(axis=1)
test_df['NoSpend'] = (test_df['TotalSpend'] == 0).astype(int)

# group size
test_df['GroupSize'] = test_df.groupby('Group')['Group'].transform('size')

# age
test_df['Age'] = test_df['Age'].fillna(test_df['Age'].median())

# CryoSleep
test_df['CryoSleep'] = test_df['CryoSleep'].map({True:1, False:0})
test_df['CryoSleep'] = test_df['CryoSleep'].fillna(0)

# interaction
test_df['Cryo_NoSpend'] = (
    (test_df['CryoSleep'] == 1) & (test_df['NoSpend'] == 1)
).astype(int)

# deck risk
test_df['DeckRisk'] = test_df['DeckNumber'].map({
    'A':0,'B':2,'C':2,'D':0,'E':-1,'F':0,'G':1,'T':-2
})

# categorical fill (same as training idea)
test_df['HomePlanet'] = test_df['HomePlanet'].fillna('Missing')
test_df['Destination'] = test_df['Destination'].fillna('Missing')
test_df['DeckNumber'] = test_df['DeckNumber'].fillna('Missing')
test_df['Side'] = test_df['Side'].map({'P': 'P', 'S': 'S'})
test_df['Side'] = test_df['Side'].fillna('Missing')
# combined feature
test_df['Deck_Side'] = test_df['DeckNumber'].astype(str) + "_" + test_df['Side'].astype(str)

# extra feature
test_df['IsChild'] = (test_df['Age'] < 13).astype(int)

test_df['Side'] = test_df['Side'].astype(str)

test_df['CabinNum'] = pd.to_numeric(
    test_df['Cabin'].str.split('/').str[1],
    errors='coerce'
)

test_df['CabinRegion'] = pd.cut(
    test_df['CabinNum'],
    bins=bin_edges,
    labels=False,
    include_lowest=True
)
# drop same columns as train
test_df = test_df.drop(columns=['PassengerId','Cabin','Name','VIP','Group'], errors='ignore')
cat_fix_cols = ['HomePlanet', 'Destination', 'DeckNumber', 'Deck_Side', 'Side']


for col in cat_fix_cols:
    test_df[col] = test_df[col].astype(str)

# =========================
# 3. ALIGN COLUMNS WITH TRAIN
# =========================
test_df = test_df[X.columns]

# =========================
# 4. PREDICT
# =========================
pred = model.predict(test_df)

# =========================
# 5. SUBMISSION
# =========================
submission = pd.DataFrame({
    'PassengerId': test_raw['PassengerId'],
    'Transported': pred.astype(bool)
})

submission.to_csv('submission4.csv', index=False)

print("SUBMISSION FILE CREATED")

SUBMISSION FILE CREATED


In [8]:
# EDA Summary: CryoSleep, NoSpend, Age and Transported
#
# Initial analysis showed that passengers with NoSpend=1 had a transport rate
# of 78.6%, compared to only 29.9% for passengers with positive expenditure.
# This suggested that the distinction between spending and non-spending passengers
# is far more important than the actual amount spent.
#
# Further investigation revealed a very strong relationship between CryoSleep
# and spending behavior:
#
#   CryoSleep = 1  --> 100% NoSpend
#   CryoSleep = 0  --> 90.5% spent money
#
# indicating that CryoSleep and NoSpend are highly related, but not identical.
#
# To determine whether NoSpend was simply a proxy for CryoSleep, the subgroup
# CryoSleep=0 and NoSpend=1 was analyzed separately. Surprisingly, these passengers
# still exhibited a transport rate of 61.6%, much higher than the baseline
# transport rate for spending passengers (~30%).
#
# Examination of this subgroup showed:
#
#   Median Age = 7 years
#   Mean Age = 10.5 years
#   75% of passengers were aged 12 or younger
#   Median GroupSize = 3
#
# suggesting that many of these passengers are children travelling with families.
# Since children are less likely to use paid onboard services, they frequently
# appear as NoSpend passengers despite not being in CryoSleep.
#
# Age analysis confirmed a strong child effect:
#
#   Adults (Age >= 13): 48.4% transported
#   Children (Age < 13): 70.0% transported
#
# Cross-analysis of CryoSleep and Age revealed two distinct behavioral regimes:
#
#   Adult + No CryoSleep  --> 30.0% transported
#   Adult + CryoSleep     --> 83.1% transported
#
#   Child + No CryoSleep  --> 68.2% transported
#   Child + CryoSleep     --> 72.4% transported
#
# This indicates that CryoSleep is an extremely strong predictor for adults,
# increasing transport probability from 30% to 83%. For children, transport
# probability is already high regardless of CryoSleep status.
#
# Conclusion:
#
# 1. CryoSleep is one of the strongest predictors of Transported.
# 2. NoSpend is not merely a duplicate of CryoSleep and captures additional
#    behavioral information.
# 3. Children naturally have a much higher transport probability than adults.
# 4. The high transport rate among CryoSleep=0 & NoSpend=1 passengers is largely
#    explained by the presence of young passengers travelling in groups.
# 5. The dataset appears to contain two major behavioral patterns:
#       - Adults: strongly influenced by CryoSleep and spending behavior.
#       - Children: inherently more likely to be transported.


In [9]:
"""# Create subgroup
subset = df[
    (df['CryoSleep'] == 0) &
    (df['NoSpend'] == 1)
]

print("="*50)
print("SIZE OF SUBGROUP")
print("="*50)
print(f"Passengers: {len(subset)}")

print("\n" + "="*50)
print("TRANSPORTED DISTRIBUTION")
print("="*50)
print(subset['Transported'].value_counts(normalize=True))

print("\n" + "="*50)
print("HOMEPLANET DISTRIBUTION")
print("="*50)
print(subset['HomePlanet'].value_counts(normalize=True))

print("\n" + "="*50)
print("DESTINATION DISTRIBUTION")
print("="*50)
print(subset['Destination'].value_counts(normalize=True))

print("\n" + "="*50)
print("SIDE DISTRIBUTION")
print("="*50)
print(subset['Side'].value_counts(normalize=True))

print("\n" + "="*50)
print("VIP DISTRIBUTION")
print("="*50)
print(subset['VIP'].value_counts(normalize=True))

print("\n" + "="*50)
print("GROUP SIZE STATISTICS")
print("="*50)
print(subset['GroupSize'].describe())

print("\n" + "="*50)
print("AGE STATISTICS")
print("="*50)
print(subset['Age'].describe())

print("\n" + "="*50)
print("HOMEPLANET vs TRANSPORTED")
print("="*50)
print(
    pd.crosstab(
        subset['HomePlanet'],
        subset['Transported'],
        normalize='index'
    )
)

print("\n" + "="*50)
print("DESTINATION vs TRANSPORTED")
print("="*50)
print(
    pd.crosstab(
        subset['Destination'],
        subset['Transported'],
        normalize='index'
    )
)

print("\n" + "="*50)
print("SIDE vs TRANSPORTED")
print("="*50)
print(
    pd.crosstab(
        subset['Side'],
        subset['Transported'],
        normalize='index'
    )
)
"""

'# Create subgroup\nsubset = df[\n    (df[\'CryoSleep\'] == 0) &\n    (df[\'NoSpend\'] == 1)\n]\n\nprint("="*50)\nprint("SIZE OF SUBGROUP")\nprint("="*50)\nprint(f"Passengers: {len(subset)}")\n\nprint("\n" + "="*50)\nprint("TRANSPORTED DISTRIBUTION")\nprint("="*50)\nprint(subset[\'Transported\'].value_counts(normalize=True))\n\nprint("\n" + "="*50)\nprint("HOMEPLANET DISTRIBUTION")\nprint("="*50)\nprint(subset[\'HomePlanet\'].value_counts(normalize=True))\n\nprint("\n" + "="*50)\nprint("DESTINATION DISTRIBUTION")\nprint("="*50)\nprint(subset[\'Destination\'].value_counts(normalize=True))\n\nprint("\n" + "="*50)\nprint("SIDE DISTRIBUTION")\nprint("="*50)\nprint(subset[\'Side\'].value_counts(normalize=True))\n\nprint("\n" + "="*50)\nprint("VIP DISTRIBUTION")\nprint("="*50)\nprint(subset[\'VIP\'].value_counts(normalize=True))\n\nprint("\n" + "="*50)\nprint("GROUP SIZE STATISTICS")\nprint("="*50)\nprint(subset[\'GroupSize\'].describe())\n\nprint("\n" + "="*50)\nprint("AGE STATISTICS")\npri

In [10]:
"""# for names
df["Title"] = df["Name"].str.extract(r' ([A-Za-z]+)\.')
df["Title"] = df["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Dr": "Rare",
    "Col": "Rare",
    "Rev": "Rare",
    "Major": "Rare",
    "Sir": "Rare",
    "Don": "Rare",
    "Jonkheer": "Rare"
})
df=df.drop('Name',axis=1)"""

<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_16/332150089.py:2: SyntaxWarning: invalid escape sequence '\.'
  df["Title"] = df["Name"].str.extract(r' ([A-Za-z]+)\.')


'# for names\ndf["Title"] = df["Name"].str.extract(r\' ([A-Za-z]+)\\.\')\ndf["Title"] = df["Title"].replace({\n    "Mlle": "Miss",\n    "Ms": "Miss",\n    "Mme": "Mrs",\n    "Lady": "Rare",\n    "Countess": "Rare",\n    "Dr": "Rare",\n    "Col": "Rare",\n    "Rev": "Rare",\n    "Major": "Rare",\n    "Sir": "Rare",\n    "Don": "Rare",\n    "Jonkheer": "Rare"\n})\ndf=df.drop(\'Name\',axis=1)'

In [11]:
"""log-transformed spending reveals a strong nonlinear relationship with Transported. Passengers with zero spending have a dramatically higher transport rate (~70%) compared to passengers with positive spending (~30%). 
The relationship appears driven primarily by the distinction between zero and non-zero spending rather than the magnitude of spending itself. 
Transported passengers exhibit a bimodal spending distribution, consistent with the previously observed CryoSleep–Spend dependency. 
This suggests that a binary NoSpend indicator may be more informative than raw spending values alone.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Create LogSpend
df['LogSpend'] = np.log1p(df['TotalSpend'])

# Create spend deciles (equal number of passengers per bin)
df['SpendDecile'] = pd.qcut(
    df['TotalSpend'],
    q=10,
    duplicates='drop'
)

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# -------------------------
# 1. Boxplot
# -------------------------
sns.boxplot(
    data=df,
    x='Transported',
    y='LogSpend',
    ax=axes[0, 0]
)

axes[0, 0].set_title('LogSpend vs Transported (Boxplot)')
axes[0, 0].set_xlabel('Transported')
axes[0, 0].set_ylabel('LogSpend')

# -------------------------
# 2. Violin Plot
# -------------------------
sns.violinplot(
    data=df,
    x='Transported',
    y='LogSpend',
    ax=axes[0, 1]
)

axes[0, 1].set_title('LogSpend vs Transported (Violin)')
axes[0, 1].set_xlabel('Transported')
axes[0, 1].set_ylabel('LogSpend')

# -------------------------
# 3. Histogram + KDE
# -------------------------
sns.histplot(
    data=df,
    x='LogSpend',
    hue='Transported',
    kde=True,
    bins=40,
    ax=axes[1, 0]
)

axes[1, 0].set_title('Distribution of LogSpend')
axes[1, 0].set_xlabel('LogSpend')

# -------------------------
# 4. Transported Rate by Spend Decile
# -------------------------
(
    df.groupby('SpendDecile', observed=False)['Transported']
      .mean()
      .plot(
          marker='o',
          linewidth=2,
          ax=axes[1, 1]
      )
)

axes[1, 1].set_title('Transported Rate Across Spend Deciles')
axes[1, 1].set_xlabel('Spend Decile')
axes[1, 1].set_ylabel('Transported Rate')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()"""

"log-transformed spending reveals a strong nonlinear relationship with Transported. Passengers with zero spending have a dramatically higher transport rate (~70%) compared to passengers with positive spending (~30%). \nThe relationship appears driven primarily by the distinction between zero and non-zero spending rather than the magnitude of spending itself. \nTransported passengers exhibit a bimodal spending distribution, consistent with the previously observed CryoSleep–Spend dependency. \nThis suggests that a binary NoSpend indicator may be more informative than raw spending values alone.\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# Create LogSpend\ndf['LogSpend'] = np.log1p(df['TotalSpend'])\n\n# Create spend deciles (equal number of passengers per bin)\ndf['SpendDecile'] = pd.qcut(\n    df['TotalSpend'],\n    q=10,\n    duplicates='drop'\n)\n\n# Create figure\nfig, axes = plt.subplots(2, 2, figsize=(16, 10))\n\n# ---------